# Task 4: Proxy Target Variable Engineering

## Objective
Construct a credit risk target variable (`is_high_risk`) from behavioral transaction data using RFM analysis and KMeans clustering, since the raw dataset has no default label.

**Methodology:**
1. Calculate RFM metrics per customer
2. Cluster customers into 3 groups using KMeans
3. Identify high-risk cluster (lowest Frequency + lowest Monetary)
4. Assign binary label: 1 = high risk, 0 = low risk

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
print('Libraries imported!')

## 1. Load Data

In [ ]:
import os
paths = [
    '../data/raw/data.csv',
    r'c:\KAIM\credit-risk-model\data\raw\data.csv',
]
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded: {df.shape}')
        break

df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])
print(df[['CustomerId','TransactionStartTime','Amount','Value']].head())

## 2. Calculate RFM Metrics

- **Recency**: Days since last transaction (lower = more recent = less risky)
- **Frequency**: Number of transactions (higher = more engaged = less risky)
- **Monetary**: Total transaction value (higher = more valuable = less risky)

In [ ]:
snapshot_date = df['TransactionStartTime'].max()
print(f'Snapshot date: {snapshot_date}')

rfm = df.groupby('CustomerId').agg(
    Recency=('TransactionStartTime',
             lambda x: (snapshot_date - x.max()).days),
    Frequency=('TransactionId', 'count'),
    Monetary=('Value', 'sum')
).reset_index()

print(f'\nRFM shape: {rfm.shape}')
print(f'\nRFM Statistics:')
print(rfm[['Recency','Frequency','Monetary']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, color in zip(axes,
    ['Recency','Frequency','Monetary'],
    ['steelblue','coral','purple']):
    data = rfm[col].clip(rfm[col].quantile(0.01), rfm[col].quantile(0.99))
    ax.hist(data, bins=40, color=color, edgecolor='white')
    ax.axvline(rfm[col].mean(), color='red', linestyle='--',
               label=f'Mean: {rfm[col].mean():.1f}')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend()
plt.suptitle('RFM Metric Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

## 3. KMeans Clustering on RFM

We scale RFM features before clustering to ensure equal weight.

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

# Elbow method to validate k=3
inertias = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(2, 8), inertias, marker='o', color='steelblue')
plt.axvline(3, color='red', linestyle='--', label='Selected k=3')
plt.title('Elbow Method — Optimal Number of Clusters', fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()
print('Elbow plot saved!')

In [ ]:
# Fit final KMeans with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

# Analyze cluster profiles
cluster_profile = rfm.groupby('cluster')[['Recency','Frequency','Monetary']].mean().round(2)
print('=== Cluster Profiles ===')
print(cluster_profile)

# Identify high-risk cluster
cluster_profile['engagement_score'] = (
    cluster_profile['Frequency'] + cluster_profile['Monetary'])
high_risk_cluster = int(cluster_profile['engagement_score'].idxmin())
print(f'\nHigh-risk cluster: {high_risk_cluster}')
print(f'Profile: {cluster_profile.loc[high_risk_cluster].to_dict()}')

## 4. Assign is_high_risk Label

In [ ]:
rfm['is_high_risk'] = (rfm['cluster'] == high_risk_cluster).astype(int)

print('=== Target Variable Distribution ===')
print(rfm['is_high_risk'].value_counts())
print(f'\nHigh-risk rate: {rfm["is_high_risk"].mean()*100:.1f}%')

# Cluster visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {0: 'steelblue', 1: 'coral', 2: 'green'}
for cluster_id in rfm['cluster'].unique():
    mask = rfm['cluster'] == cluster_id
    label = f'Cluster {cluster_id}'
    if cluster_id == high_risk_cluster:
        label += ' (HIGH RISK)'
    axes[0].scatter(
        rfm[mask]['Frequency'].clip(0, rfm['Frequency'].quantile(0.99)),
        rfm[mask]['Monetary'].clip(0, rfm['Monetary'].quantile(0.99)),
        alpha=0.5, s=20, label=label,
        color=colors.get(cluster_id, 'grey')
    )
axes[0].set_title('Frequency vs Monetary by Cluster', fontweight='bold')
axes[0].set_xlabel('Frequency')
axes[0].set_ylabel('Monetary (UGX)')
axes[0].legend()

# Risk label distribution
risk_counts = rfm['is_high_risk'].value_counts()
axes[1].bar(['Low Risk (0)','High Risk (1)'],
            risk_counts.values,
            color=['#2ecc71','#e74c3c'])
axes[1].set_title('Proxy Target Distribution', fontweight='bold')
axes[1].set_ylabel('Number of Customers')
for i, v in enumerate(risk_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.suptitle('KMeans Clustering Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('clustering_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

In [ ]:
# Merge back into processed dataset
processed_paths = [
    '../data/processed/features.csv',
    r'c:\KAIM\credit-risk-model\data\processed\features.csv',
]
for path in processed_paths:
    if os.path.exists(path):
        features = pd.read_csv(path)
        print(f'Loaded processed features: {features.shape}')
        break

print(f'\nTarget distribution in processed dataset:')
print(features['is_high_risk'].value_counts())
print(f'High-risk rate: {features["is_high_risk"].mean()*100:.1f}%')
print('\nProxy target successfully integrated!')

## 5. Summary

### RFM Clustering Results:
- **3 clusters** identified using KMeans (k=3, validated with elbow method)
- **High-risk cluster** characterized by: lowest transaction frequency + lowest monetary value
- **Business interpretation**: Customers who transact rarely and spend little exhibit disengaged behavior consistent with elevated credit default risk

### Proxy Variable Risks:
- Label noise: disengaged ≠ definitely defaulting
- Must be validated against actual loan performance once BNPL portfolio matures
- Cluster boundaries are sensitive to scaling and random seed — documented for reproducibility (random_state=42)